In [ ]:
# Data Visualization Libraries
from pathlib import Path
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print("All libraries imported successfully.")

In [ ]:

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "raw"
METADATA_PATH = DATA_DIR / "metadata.csv"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Metadata Path: {METADATA_PATH}")

In [ ]:
# Load Dataset Metadata

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_PATH}")

metadata = pd.read_csv(METADATA_PATH)

print("Metadata loaded successfully")
print("--------------------------------")
print(f"Dataset shape: {metadata.shape}")

print("\nColumns:")
print(metadata.columns.tolist())

display(metadata.head())

In [ ]:
# Class Distribution Visualization

plt.figure(figsize=(8, 5))

class_counts = metadata["Label"].value_counts()

sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)

plt.title("Chest X-ray Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

for i, value in enumerate(class_counts.values):
    plt.text(
        i,
        value + 500,
        f"{value:,}",
        ha="center"
    )

plt.show()

print("Class distribution:")
print(class_counts)

In [ ]:
# Finding Labels Distribution

# Split multiple findings into separate rows
findings = (
    metadata["Finding Labels"]
    .str.split("|")
    .explode()
)

finding_counts = findings.value_counts()

print("Number of unique findings:", len(finding_counts))

display(finding_counts.head(15))


# Plot top 15 findings

plt.figure(figsize=(10, 6))

sns.barplot(
    y=finding_counts.head(15).index,
    x=finding_counts.head(15).values
)

plt.title("Top 15 Chest X-ray Findings")
plt.xlabel("Number of Images")
plt.ylabel("Finding")

plt.show()

In [ ]:
# Dataset Split Distribution

split_counts = metadata["Split"].value_counts()

print("Dataset Split Distribution")
print("--------------------------")
print(split_counts)


plt.figure(figsize=(7, 5))

sns.barplot(
    x=split_counts.index,
    y=split_counts.values
)

plt.title("Train, Validation and Test Split Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")

for i, value in enumerate(split_counts.values):
    plt.text(
        i,
        value + 500,
        f"{value:,}",
        ha="center"
    )

plt.show()

In [ ]:
# Prepare Features for PCA/t-SNE Visualization

from sklearn.preprocessing import LabelEncoder

# Encode labels
encoder = LabelEncoder()

metadata["Label_encoded"] = encoder.fit_transform(metadata["Label"])

print("Label encoding:")
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

# Select categorical features from findings
finding_features = (
    metadata["Finding Labels"]
    .str.get_dummies(sep="|")
)

# Combine features
feature_data = pd.concat(
    [
        finding_features,
        metadata[["Label_encoded"]]
    ],
    axis=1
)

print("\nFeature matrix shape:")
print(feature_data.shape)

display(feature_data.head())

In [ ]:
# PCA Feature Space Visualization

# Separate features and labels
X = feature_data.drop("Label_encoded", axis=1)
y = feature_data["Label_encoded"]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["Label"] = metadata["Label"].values

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

display(pca_df.head())

In [ ]:
# PCA Feature Space Visualization

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Label",
    alpha=0.6
)

plt.title("PCA Visualization of Chest X-ray Feature Space")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.legend(title="Class")
plt.show()

In [ ]:
# Feature Correlation Heatmap

plt.figure(figsize=(12, 8))

correlation_matrix = feature_data.corr()

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Chest X-ray Features")
plt.xlabel("Features")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
# 3D Scatter Plot of Top 3 Features

# Calculate feature variance
feature_variance = X.var().sort_values(ascending=False)

top_features = feature_variance.head(3).index.tolist()

print("Top 3 features selected:")
print(top_features)


plot_3d_data = feature_data[top_features].copy()

plot_3d_data["Label"] = metadata["Label"]


fig = px.scatter_3d(
    plot_3d_data,
    x=top_features[0],
    y=top_features[1],
    z=top_features[2],
    color="Label",
    title=f"3D Feature Interaction: {top_features}",
    opacity=0.6
)

fig.show()

## Visualization Summary

This notebook performs exploratory data analysis and feature visualization of the chest X-ray dataset.

Completed analyses:
- Class distribution analysis
- Disease finding frequency analysis
- Dataset split visualization
- PCA feature space visualization
- Feature correlation analysis
- 3D feature interaction visualization

Model-specific visualizations such as feature importance and model comparison charts will be generated after model training.